# 03 — Metodologia e Experimentos Preditivos

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia Técnico](./reference_materials/03_metodologia_experimentos.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_metodologia_experimentos.pdf)
- [Documentação da área](./README.pt-BR.md)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Formular hipótese, escolher baseline, planejar treino/validação/teste, evitar contaminação e selecionar divisões compatíveis com o problema.

O conjunto de teste representa avaliação final e não deve guiar a seleção de hiperparâmetros. `train_test_split` é útil para observações independentes; dados de uma pessoa/grupo ou dados temporais requerem outro esquema. A validação cruzada pertence ao treino e pode ser aninhada em problemas exigentes.


## 2. Fundamentos matemáticos

Para um erro médio $\widehat R_{\mathrm{teste}}=\frac1m\sum_{i=1}^m L(y_i,\hat f(x_i))$, a qualidade da estimativa depende de como as observações de teste representam o uso futuro. Viés de seleção surge quando escolhemos o modelo após inspecionar repetidamente o teste.

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
X,y=load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
cv=StratifiedKFold(5,shuffle=True,random_state=42)
model=LogisticRegression(max_iter=2000,solver='liblinear')
print('Baseline no treino:',round(DummyClassifier(strategy='most_frequent').fit(Xtr,ytr).score(Xtr,ytr),3))
print('Validação cruzada (somente treino):',cross_val_score(model,Xtr,ytr,cv=cv,scoring='balanced_accuracy').round(3))
model.fit(Xtr,ytr)
print('Teste final:',round(model.score(Xte,yte),3))

Baseline no treino: 0.626
Validação cruzada (somente treino): [0.941 0.93  0.968 0.932 0.944]
Teste final: 0.956


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
from sklearn.model_selection import GroupShuffleSplit, TimeSeriesSplit
import numpy as np
X=np.arange(24).reshape(-1,1); groups=np.repeat(np.arange(8),3)
train_idx,test_idx=next(GroupShuffleSplit(test_size=.25,n_splits=1,random_state=42).split(X,groups=groups))
print('Grupos disjuntos:',set(groups[train_idx]).isdisjoint(set(groups[test_idx])))
print('Folds temporais (índices):',[(a[-1],b[0]) for a,b in TimeSeriesSplit(n_splits=3).split(X)])

Grupos disjuntos: True
Folds temporais (índices): [(np.int64(5), np.int64(6)), (np.int64(11), np.int64(12)), (np.int64(17), np.int64(18))]


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: esquema de avaliação apropriado
cenarios={'Novos clientes independentes':'holdout estratificado para classificação', 'Várias linhas por paciente':'divisão por paciente/grupo', 'Prever mês seguinte':'divisão cronológica'}
for k,v in cenarios.items(): print(k,'=>',v)

Novos clientes independentes => holdout estratificado para classificação
Várias linhas por paciente => divisão por paciente/grupo
Prever mês seguinte => divisão cronológica


## 6. Interpretação, limitações e erros comuns

A estratificação ajuda a manter proporções, mas não remove dependência entre observações. A divisão temporal deve respeitar horizonte, *lag* e, quando necessário, intervalo de segurança (*gap*). O objetivo e o regime de avaliação precisam ser declarados antes do primeiro `fit`.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- Treino ajusta; validação compara; teste estima desempenho final.
- Grupos correlacionados exigem `GroupKFold` ou equivalente.
- Séries temporais: `TimeSeriesSplit` ou holdout cronológico.
- Reutilizar teste em iterações sucessivas contamina decisões.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular


**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** vazamento e divisão. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **vazamento e divisão** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Vazamento e validação temporal

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.


In [4]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
idx = np.arange(15)
for rodada, (treino, teste) in enumerate(TimeSeriesSplit(n_splits=3).split(idx), 1):
    assert max(treino) < min(teste)
    print(f"Divisão {rodada}: treino={treino.tolist()} | teste={teste.tolist()}")

Divisão 1: treino=[0, 1, 2, 3, 4, 5] | teste=[6, 7, 8]
Divisão 2: treino=[0, 1, 2, 3, 4, 5, 6, 7, 8] | teste=[9, 10, 11]
Divisão 3: treino=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] | teste=[12, 13, 14]


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **vazamento e divisão** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
